# 🔮 Prediction — Fake Product Review Detection

This notebook loads the **saved model and TF-IDF vectorizer** (no retraining!) and lets you test predictions on any review text.

## Prerequisites

Make sure you've already run `train.ipynb` (or `python src/train.py`) to generate:
- `models/logistic_regression_model.pkl`
- `models/tfidf_vectorizer.pkl`

---

## 1. Import Libraries & Setup Paths

In [1]:
import os
import re
import joblib
import pandas as pd

# ── Paths ──
PROJECT_DIR = os.path.abspath("..")
MODEL_PATH = os.path.join(PROJECT_DIR, "models", "logistic_regression_model.pkl")
VECTORIZER_PATH = os.path.join(PROJECT_DIR, "models", "tfidf_vectorizer.pkl")

print(f"📁 Project directory: {PROJECT_DIR}")
print(f"📁 Model path:       {MODEL_PATH}")
print(f"📁 Vectorizer path:  {VECTORIZER_PATH}")

📁 Project directory: c:\Users\ayush\OneDrive\Desktop\Project\fake-product-review-detection
📁 Model path:       c:\Users\ayush\OneDrive\Desktop\Project\fake-product-review-detection\models\logistic_regression_model.pkl
📁 Vectorizer path:  c:\Users\ayush\OneDrive\Desktop\Project\fake-product-review-detection\models\tfidf_vectorizer.pkl


## 2. Define Text Cleaning Function

We must apply the **exact same text cleaning** as during training. If we don't, the TF-IDF vectorizer won't match words correctly.

In [2]:
def clean_text(text):
    """Clean a single review text string (same as training)."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ clean_text() defined")

✅ clean_text() defined


## 3. Load Saved Model & Vectorizer

We load the pre-trained model and vectorizer from disk. **No retraining happens here.**

In [3]:
# ── Check files exist ──
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Model not found at: {MODEL_PATH}\n   Run train.ipynb first!")
if not os.path.exists(VECTORIZER_PATH):
    raise FileNotFoundError(f"❌ Vectorizer not found at: {VECTORIZER_PATH}\n   Run train.ipynb first!")

# ── Load ──
print("📦 Loading model and vectorizer...")
model = joblib.load(MODEL_PATH)
vectorizer = joblib.load(VECTORIZER_PATH)
print("✅ Model and vectorizer loaded successfully!")
print(f"   Model type: {type(model).__name__}")
print(f"   Vocabulary size: {len(vectorizer.vocabulary_):,} terms")

📦 Loading model and vectorizer...
✅ Model and vectorizer loaded successfully!
   Model type: LogisticRegression
   Vocabulary size: 15,000 terms


## 4. Prediction Function

This function takes a review text, cleans it, vectorizes it with TF-IDF, and returns the prediction with confidence.

In [4]:
def predict_review(review_text):
    """
    Predict whether a review is Fake or Genuine.
    
    Args:
        review_text (str): The review to classify.
    
    Returns:
        dict: Prediction results with label, confidence, and probabilities.
    """
    cleaned = clean_text(review_text)
    
    if not cleaned.strip():
        return {'label': 'Unknown', 'confidence': 0.0, 'fake_prob': 0.0, 'genuine_prob': 0.0}
    
    # Transform using saved vectorizer
    text_tfidf = vectorizer.transform([cleaned])
    
    # Predict
    prediction = model.predict(text_tfidf)[0]
    probabilities = model.predict_proba(text_tfidf)[0]
    
    label = "Fake" if prediction == 1 else "Genuine"
    confidence = probabilities[prediction] * 100
    
    return {
        'label': label,
        'confidence': confidence,
        'fake_prob': probabilities[1] * 100,
        'genuine_prob': probabilities[0] * 100,
        'cleaned_text': cleaned,
    }

print("✅ predict_review() defined")

✅ predict_review() defined


---

## 5. Test Predictions

Let's test the model with various sample reviews. Edit the reviews below or add your own!

### 5.1 Single Review Prediction

In [5]:
# ── Try your own review! ──
review = "This product is absolutely amazing and works perfectly. Highly recommended!"

result = predict_review(review)

print("┌─────────────────────────────────────────────┐")
print(f"│ 📝 Review: {review[:50]}...")
print(f"│")
if result['label'] == 'Fake':
    print(f"│ 🚨 Prediction:  FAKE REVIEW")
else:
    print(f"│ ✅ Prediction:  GENUINE REVIEW")
print(f"│ 📊 Confidence:  {result['confidence']:.2f}%")
print(f"│ 📊 Fake prob:   {result['fake_prob']:.2f}%")
print(f"│ 📊 Genuine prob: {result['genuine_prob']:.2f}%")
print("└─────────────────────────────────────────────┘")

┌─────────────────────────────────────────────┐
│ 📝 Review: This product is absolutely amazing and works perfe...
│
│ ✅ Prediction:  GENUINE REVIEW
│ 📊 Confidence:  90.18%
│ 📊 Fake prob:   9.82%
│ 📊 Genuine prob: 90.18%
└─────────────────────────────────────────────┘


### 5.2 Batch Predictions

Test multiple reviews at once and see results in a table.

In [6]:
# ── Sample reviews to test ──
test_reviews = [
    "This product is absolutely amazing and works perfectly. Highly recommended!",
    "Terrible quality. Broke after one day. Complete waste of money. Do not buy.",
    "I bought this product and it was great great great amazing wonderful perfect love it so much best ever",
    "The build quality is decent for the price point. Battery life could be better but overall satisfactory for daily use.",
    "Wow this is the best thing I ever bought amazing incredible fantastic wonderful superb outstanding",
    "I've been using this for about 3 months now. The stitching on the left side started coming undone after week 6, but customer service sent a replacement quickly.",
    "Five stars! Best product! Amazing quality! Perfect! Love it! Recommend to everyone! Buy now!",
    "It works as described. Nothing fancy but gets the job done. Shipping was fast.",
]

# ── Predict all ──
results = []
for review in test_reviews:
    r = predict_review(review)
    results.append({
        'Review': review[:80] + ('...' if len(review) > 80 else ''),
        'Prediction': r['label'],
        'Confidence': f"{r['confidence']:.1f}%",
        'Fake %': f"{r['fake_prob']:.1f}%",
        'Genuine %': f"{r['genuine_prob']:.1f}%",
    })

results_df = pd.DataFrame(results)
print("📊 Batch Prediction Results:\n")
results_df

📊 Batch Prediction Results:



,Review,Prediction,Confidence,Fake %,Genuine %
0,This product is absolutely amazing and works p...,Genuine,90.2%,9.8%,90.2%
1,Terrible quality. Broke after one day. Complet...,Genuine,90.3%,9.7%,90.3%
2,I bought this product and it was great great g...,Genuine,72.4%,27.6%,72.4%
3,The build quality is decent for the price poin...,Genuine,92.9%,7.1%,92.9%
4,Wow this is the best thing I ever bought amazi...,Genuine,73.0%,27.0%,73.0%
5,I've been using this for about 3 months now. T...,Genuine,91.8%,8.2%,91.8%
6,Five stars! Best product! Amazing quality! Per...,Genuine,90.3%,9.7%,90.3%
7,It works as described. Nothing fancy but gets ...,Genuine,90.8%,9.2%,90.8%


### 5.3 Try Your Own Review

Change the `my_review` variable below to test any review you want!

In [7]:
# ═══════════════════════════════════════════════
# ✏️  EDIT THIS REVIEW TO TEST YOUR OWN TEXT!
# ═══════════════════════════════════════════════

my_review = "Type your review here and run this cell!"

# ═══════════════════════════════════════════════

result = predict_review(my_review)

print(f"📝 Review: {my_review}")
print(f"\n🔍 Cleaned: {result['cleaned_text']}")
print(f"\n{'🚨 FAKE' if result['label'] == 'Fake' else '✅ GENUINE'} — {result['confidence']:.2f}% confidence")
print(f"   Fake probability:    {result['fake_prob']:.2f}%")
print(f"   Genuine probability: {result['genuine_prob']:.2f}%")

📝 Review: Type your review here and run this cell!

🔍 Cleaned: type your review here and run this cell

✅ GENUINE — 92.79% confidence
   Fake probability:    7.21%
   Genuine probability: 92.79%


---

## ✅ Prediction Notebook Complete!

You can use this notebook to:
- Test individual reviews (Section 5.1)
- Run batch predictions on multiple reviews (Section 5.2)
- Try your own reviews (Section 5.3)

For the web interface, run: `streamlit run app.py`